In [ ]:

#Imports
import numpy as np
import matplotlib.pyplot as plt
import csv
from pathlib import Path

In [ ]:
## Decisons
class Config:
    def __init__(self):
        self.file_path = Path('data.csv') ## This is the path to the CSV file. Change this to the path of your CSV file.
        self.plot_leading_edge = False ## Set to True if you want to plot the leading edge of the data.


In [ ]:
def Create_convex_hull(densities: np.ndarray, energies: np.ndarray):
    """Create the lower leading-edge envelope, tethered to the graph edges.

    The boundary should run from the left edge to the right edge without looping
    back on itself. To enforce that, we add explicit edge anchors and then build
    the monotone lower hull from left to right using the real data only.
    """
    densities = np.asarray(densities, dtype=float)
    energies = np.asarray(energies, dtype=float)

    valid = np.isfinite(densities) & np.isfinite(energies)
    densities = densities[valid]
    energies = energies[valid]

    if densities.size == 0:
        return np.array([], dtype=float), np.array([], dtype=float)
    if densities.size == 1:
        return np.array([energies[0], energies[0]], dtype=float), np.array([densities[0], densities[0]], dtype=float)

    x_min = densities.min()
    x_max = densities.max()
    y_max = energies.max()
    pad_x = max(0.05, 0.05 * (x_max - x_min)) if x_max > x_min else 0.05
    pad_y = max(10.0, 0.05 * abs(y_max))

    left_anchor = np.array([x_min - pad_x, y_max + pad_y], dtype=float)
    right_anchor = np.array([x_max + pad_x, y_max + pad_y], dtype=float)

    unique_densities = np.unique(densities)
    min_energies = np.array([
        energies[densities == density].min() for density in unique_densities
    ], dtype=float)
    points = np.column_stack((unique_densities, min_energies))
    points = np.vstack([points, left_anchor, right_anchor])

    ordered = points[np.lexsort((points[:, 1], points[:, 0]))]
    lower = []
    for point in ordered:
        while len(lower) >= 2:
            a = lower[-2]
            b = lower[-1]
            cross = (b[0] - a[0]) * (point[1] - a[1]) - (b[1] - a[1]) * (point[0] - a[0])
            if cross <= 0:
                lower.pop()
            else:
                break
        lower.append(point)

    lower = np.asarray(lower, dtype=float)
    lower = lower[np.argsort(lower[:, 0])]

    x_dense = np.linspace(lower[:, 0].min(), lower[:, 0].max(), max(200, len(lower) * 10))
    y_dense = np.interp(x_dense, lower[:, 0], lower[:, 1])
    return y_dense, x_dense

In [ ]:

config=Config()
with open(config.file_path, mode='r', newline='') as csvfile:
    reader = csv.reader(csvfile)
    data = list(reader)
file_name, origin_of_structure, energy_per_atom, density, formula_units, stoichiometry, has_cyanurate_structure, has_octahedral_indium, has_InN6_InS6_structure, in_low_energy_folder = zip(*data)

fig = plt.figure(figsize=(10, 6))

if config.plot_leading_edge:
    energies, densities = Create_convex_hull(density, energy_per_atom)
    plt.plot(densities, energies, color='k', label='Leading Edge', zorder= 10)
    plt.plot(density, energy_per_atom +30, 'r', label='Leading edge + 30meV/atom ', zorder= 10)

